# Co-Training Smoke Test

Quick end-to-end test of the LG-CoTrain pipeline across **all 3 modalities** and **both tasks** using Llama pseudo-labels.

Uses minimal epochs to verify the pipeline runs without errors — not for real results.

**Experiments:** 2 tasks × 3 modalities = **6 runs** (budget=5, seed_set=1)

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import time
import pandas as pd
from lg_cotrain.config import LGCoTrainConfig
from lg_cotrain.trainer import LGCoTrainer

In [2]:
# Smoke test config: minimal epochs + small unlabeled set for fast validation
SMOKE_PARAMS = dict(
    weight_gen_epochs=2,
    cotrain_epochs=2,
    finetune_max_epochs=3,
    finetune_patience=2,
    batch_size=16,
    lr=2e-5,
    pseudo_label_source="llama-3.2-11b",
    max_unlabeled_samples=200,  # cap D_LG size for speed
)

DATA_ROOT = os.path.abspath("../data")
RESULTS_ROOT = os.path.abspath("../results/cotrain_smoke_test")

TASKS = ["informative", "humanitarian"]
MODALITIES = ["text_only", "image_only", "text_image"]
BUDGET = 5
SEED_SET = 1

print(f"Data root:    {DATA_ROOT}")
print(f"Results root: {RESULTS_ROOT}")
print(f"Experiments:  {len(TASKS)} tasks x {len(MODALITIES)} modalities = {len(TASKS) * len(MODALITIES)} runs")
print(f"Budget: {BUDGET}, Seed set: {SEED_SET}, Max unlabeled: {SMOKE_PARAMS['max_unlabeled_samples']}")

Data root:    D:\Workspace\Cotrain_CrisisMMD\data
Results root: D:\Workspace\Cotrain_CrisisMMD\results\cotrain_smoke_test
Experiments:  2 tasks x 3 modalities = 6 runs
Budget: 5, Seed set: 1, Max unlabeled: 200


## Run All Experiments

Each cell runs one (task, modality) combination with budget=5, seed_set=1.

Pipeline per experiment:
1. Phase 1: Weight generation (2 epochs)
2. Phase 2: Co-training (2 epochs)
3. Phase 3: Fine-tuning (up to 3 epochs, patience=2)
4. Ensemble evaluation on dev + test

In [3]:
all_results = []

def run_smoke_test(task, modality):
    """Run a single smoke-test experiment and return results."""
    print(f"\n{'='*60}")
    print(f"Task: {task} | Modality: {modality} | Budget: {BUDGET} | Seed: {SEED_SET}")
    print(f"{'='*60}")
    
    config = LGCoTrainConfig(
        task=task,
        modality=modality,
        budget=BUDGET,
        seed_set=SEED_SET,
        data_root=DATA_ROOT,
        results_root=RESULTS_ROOT,
        **SMOKE_PARAMS,
    )
    
    trainer = LGCoTrainer(config)
    
    start = time.time()
    results = trainer.run()
    elapsed = time.time() - start
    
    print(f"\nCompleted in {elapsed:.1f}s")
    print(f"  Test Macro-F1:   {results['test_macro_f1']:.4f}")
    print(f"  Test Error Rate: {results['test_error_rate']:.2f}%")
    print(f"  Dev Macro-F1:    {results['dev_macro_f1']:.4f}")
    
    all_results.append(results)
    return results

### Informative — text_only (BERTweet)

In [4]:
run_smoke_test("informative", "text_only")


Task: informative | Modality: text_only | Budget: 5 | Seed: 1


emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0
2026-04-01 14:01:26,644 - lg_cotrain - INFO - Starting LG-CoTrain: task=informative, modality=text_only, budget=5, seed_set=1
2026-04-01 14:01:26,680 - lg_cotrain - INFO - Detected 2 classes: ['informative', 'not_informative']
2026-04-01 14:01:26,685 - lg_cotrain - INFO - Capped unlabeled set to 200 samples
2026-04-01 14:01:26,688 - lg_cotrain - INFO - D_l1: 6, D_l2: 4, D_LG: 200
2026-04-01 14:01:26,690 - lg_cotrain - INFO - === Phase 1: Weight Generation ===
2026-04-01 14:01:29,234 - lg_cotrain - INFO - Phase 1 epoch 1/2: mean_prob1=0.4807, mean_prob2=0.4866
2026-04-01 14:01:30,005 - lg_cotrain - INFO - Phase 1 epoch 2/2: mean_prob1=0.4855, mean_prob2=0.4875
2026-04-01 14:01:30,006 - lg_cotrain - INFO - === Phase 2: Co-Training ===
2026-04-01 14:01:30,758 - lg_cotrain - INFO - LR scheduler: total_steps_1=29, warmup=2; total_steps_2=29, warmup=2
2026-04-01 14:01:30,759 - 


Completed in 49.0s
  Test Macro-F1:   0.4017
  Test Error Rate: 32.86%
  Dev Macro-F1:    0.4017


{'task': 'informative',
 'modality': 'text_only',
 'budget': 5,
 'seed_set': 1,
 'test_error_rate': np.float64(32.85528031290743),
 'test_macro_f1': 0.4017160686427457,
 'test_ece': 0.20303160057074085,
 'test_per_class_f1': [0.8034321372854915, 0.0],
 'dev_error_rate': np.float64(32.86713286713287),
 'dev_macro_f1': 0.401673640167364,
 'dev_ece': 0.20326662226535686,
 'stopping_strategy': 'baseline',
 'phase1_seed_strategy': 'last',
 'phase1_best_epoch': None,
 'lambda1_mean': 0.8192104251495231,
 'lambda1_std': 0.17915942806425758,
 'lambda2_mean': 0.4642792695067773,
 'lambda2_std': 0.17469879539316516}

In [5]:
run_smoke_test("informative", "image_only")

2026-04-01 14:02:15,772 - lg_cotrain - INFO - Starting LG-CoTrain: task=informative, modality=image_only, budget=5, seed_set=1
2026-04-01 14:02:15,810 - lg_cotrain - INFO - Detected 2 classes: ['informative', 'not_informative']
2026-04-01 14:02:15,815 - lg_cotrain - INFO - Capped unlabeled set to 200 samples
2026-04-01 14:02:15,820 - lg_cotrain - INFO - D_l1: 6, D_l2: 4, D_LG: 200
2026-04-01 14:02:15,821 - lg_cotrain - INFO - === Phase 1: Weight Generation ===



Task: informative | Modality: image_only | Budget: 5 | Seed: 1


2026-04-01 14:02:22,115 - lg_cotrain - INFO - Phase 1 epoch 1/2: mean_prob1=0.5394, mean_prob2=0.5302
2026-04-01 14:02:27,732 - lg_cotrain - INFO - Phase 1 epoch 2/2: mean_prob1=0.6207, mean_prob2=0.5701
2026-04-01 14:02:27,733 - lg_cotrain - INFO - === Phase 2: Co-Training ===
2026-04-01 14:02:28,822 - lg_cotrain - INFO - LR scheduler: total_steps_1=29, warmup=2; total_steps_2=29, warmup=2
2026-04-01 14:02:28,823 - lg_cotrain - INFO - Phase 1 done -> seeded Phase 2 trackers. lambda1: mean=0.6207, range=[0.1023, 0.9709]
2026-04-01 14:02:28,823 - lg_cotrain - INFO - Phase 1 done -> seeded Phase 2 trackers. lambda2: mean=0.5701, range=[0.0916, 0.9204]
2026-04-01 14:02:58,773 - lg_cotrain - INFO - Phase 2 epoch 1/2: loss1=0.2434, loss2=0.3088, dev_macro_f1=0.7897, dev_err=17.29%
2026-04-01 14:03:28,683 - lg_cotrain - INFO - Phase 2 epoch 2/2: loss1=0.0491, loss2=0.1515, dev_macro_f1=0.7414, dev_err=19.14%
2026-04-01 14:03:28,684 - lg_cotrain - INFO - === Phase 3: Fine-Tuning ===
2026-04-0


Completed in 179.4s
  Test Macro-F1:   0.7819
  Test Error Rate: 16.82%
  Dev Macro-F1:    0.7694


{'task': 'informative',
 'modality': 'image_only',
 'budget': 5,
 'seed_set': 1,
 'test_error_rate': np.float64(16.81877444589309),
 'test_macro_f1': 0.7818716931216931,
 'test_ece': 0.05284149915640478,
 'test_per_class_f1': [0.8862433862433863, 0.6775],
 'dev_error_rate': np.float64(17.546090273363003),
 'dev_macro_f1': 0.7694342123711078,
 'dev_ece': 0.06186818621830054,
 'stopping_strategy': 'baseline',
 'phase1_seed_strategy': 'last',
 'phase1_best_epoch': None,
 'lambda1_mean': 0.9844087880182693,
 'lambda1_std': 0.0737832070644917,
 'lambda2_mean': 0.6089730502815328,
 'lambda2_std': 0.16072950684073622}

In [6]:
run_smoke_test("informative", "text_image")


Task: informative | Modality: text_image | Budget: 5 | Seed: 1


emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0
2026-04-01 14:05:15,670 - lg_cotrain - INFO - Starting LG-CoTrain: task=informative, modality=text_image, budget=5, seed_set=1
2026-04-01 14:05:15,740 - lg_cotrain - INFO - Detected 2 classes: ['informative', 'not_informative']
2026-04-01 14:05:15,744 - lg_cotrain - INFO - Capped unlabeled set to 200 samples
2026-04-01 14:05:15,748 - lg_cotrain - INFO - D_l1: 6, D_l2: 4, D_LG: 200
2026-04-01 14:05:15,750 - lg_cotrain - INFO - === Phase 1: Weight Generation ===
2026-04-01 14:05:23,876 - lg_cotrain - INFO - Phase 1 epoch 1/2: mean_prob1=0.5583, mean_prob2=0.5616
2026-04-01 14:05:29,735 - lg_cotrain - INFO - Phase 1 epoch 2/2: mean_prob1=0.5871, mean_prob2=0.5935
2026-04-01 14:05:29,736 - lg_cotrain - INFO - === Phase 2: Co-Training ===
2026-04-01 14:05:31,536 - lg_cotrain - INFO - LR scheduler: total_steps_1=29, warmup=2; total_steps_2=29, warmup=2
2026-04-01 14:05:31,538 -


Completed in 198.8s
  Test Macro-F1:   0.5208
  Test Error Rate: 28.88%
  Dev Macro-F1:    0.5100


{'task': 'informative',
 'modality': 'text_image',
 'budget': 5,
 'seed_set': 1,
 'test_error_rate': np.float64(28.878748370273797),
 'test_macro_f1': 0.5207828685104441,
 'test_ece': 0.19998468138092973,
 'test_per_class_f1': [0.8228708516593363, 0.21869488536155202],
 'dev_error_rate': np.float64(29.434202161474886),
 'dev_macro_f1': 0.5099896184668837,
 'dev_ece': 0.21238603549479534,
 'stopping_strategy': 'baseline',
 'phase1_seed_strategy': 'last',
 'phase1_best_epoch': None,
 'lambda1_mean': 0.956315793182674,
 'lambda1_std': 0.12675615501541915,
 'lambda2_mean': 0.6381018992451595,
 'lambda2_std': 0.15061020731035155}

### Humanitarian Task

In [7]:
run_smoke_test("humanitarian", "text_only")


Task: humanitarian | Modality: text_only | Budget: 5 | Seed: 1


emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0
2026-04-01 14:08:34,927 - lg_cotrain - INFO - Starting LG-CoTrain: task=humanitarian, modality=text_only, budget=5, seed_set=1
2026-04-01 14:08:34,960 - lg_cotrain - INFO - Detected 5 classes: ['affected_individuals', 'infrastructure_and_utility_damage', 'not_humanitarian', 'other_relevant_information', 'rescue_volunteering_or_donation_effort']
2026-04-01 14:08:34,962 - lg_cotrain - INFO - Capped unlabeled set to 200 samples
2026-04-01 14:08:34,966 - lg_cotrain - INFO - D_l1: 15, D_l2: 10, D_LG: 200
2026-04-01 14:08:34,968 - lg_cotrain - INFO - === Phase 1: Weight Generation ===
2026-04-01 14:08:36,646 - lg_cotrain - INFO - Phase 1 epoch 1/2: mean_prob1=0.1938, mean_prob2=0.1906
2026-04-01 14:08:37,548 - lg_cotrain - INFO - Phase 1 epoch 2/2: mean_prob1=0.1957, mean_prob2=0.1909
2026-04-01 14:08:37,548 - lg_cotrain - INFO - === Phase 2: Co-Training ===
2026-04-01 14:08:38


Completed in 35.5s
  Test Macro-F1:   0.1382
  Test Error Rate: 47.23%
  Dev Macro-F1:    0.1372


{'task': 'humanitarian',
 'modality': 'text_only',
 'budget': 5,
 'seed_set': 1,
 'test_error_rate': np.float64(47.225130890052355),
 'test_macro_f1': 0.13817683344756682,
 'test_ece': 0.1060795506881794,
 'test_per_class_f1': [0.0, 0.0, 0.6908841672378341, 0.0, 0.0],
 'dev_error_rate': np.float64(47.795591182364724),
 'dev_macro_f1': 0.1371955233706386,
 'dev_ece': 0.09833130094116338,
 'stopping_strategy': 'baseline',
 'phase1_seed_strategy': 'last',
 'phase1_best_epoch': None,
 'lambda1_mean': 0.3488077308765129,
 'lambda1_std': 0.1126189783728852,
 'lambda2_mean': 0.20150824665468886,
 'lambda2_std': 0.04668656521688159}

In [8]:
run_smoke_test("humanitarian", "image_only")

2026-04-01 14:09:10,488 - lg_cotrain - INFO - Starting LG-CoTrain: task=humanitarian, modality=image_only, budget=5, seed_set=1
2026-04-01 14:09:10,511 - lg_cotrain - INFO - Detected 5 classes: ['affected_individuals', 'infrastructure_and_utility_damage', 'not_humanitarian', 'other_relevant_information', 'rescue_volunteering_or_donation_effort']
2026-04-01 14:09:10,513 - lg_cotrain - INFO - Capped unlabeled set to 200 samples
2026-04-01 14:09:10,516 - lg_cotrain - INFO - D_l1: 15, D_l2: 10, D_LG: 200
2026-04-01 14:09:10,517 - lg_cotrain - INFO - === Phase 1: Weight Generation ===



Task: humanitarian | Modality: image_only | Budget: 5 | Seed: 1


2026-04-01 14:09:16,841 - lg_cotrain - INFO - Phase 1 epoch 1/2: mean_prob1=0.2416, mean_prob2=0.2458
2026-04-01 14:09:22,502 - lg_cotrain - INFO - Phase 1 epoch 2/2: mean_prob1=0.2449, mean_prob2=0.2711
2026-04-01 14:09:22,504 - lg_cotrain - INFO - === Phase 2: Co-Training ===
2026-04-01 14:09:23,592 - lg_cotrain - INFO - LR scheduler: total_steps_1=29, warmup=2; total_steps_2=29, warmup=2
2026-04-01 14:09:23,592 - lg_cotrain - INFO - Phase 1 done -> seeded Phase 2 trackers. lambda1: mean=0.2449, range=[0.0372, 0.6505]
2026-04-01 14:09:23,593 - lg_cotrain - INFO - Phase 1 done -> seeded Phase 2 trackers. lambda2: mean=0.2711, range=[0.0512, 0.6350]
2026-04-01 14:09:45,238 - lg_cotrain - INFO - Phase 2 epoch 1/2: loss1=0.2717, loss2=0.2925, dev_macro_f1=0.4855, dev_err=28.56%
2026-04-01 14:10:07,048 - lg_cotrain - INFO - Phase 2 epoch 2/2: loss1=0.0753, loss2=0.1573, dev_macro_f1=0.5120, dev_err=26.75%
2026-04-01 14:10:07,049 - lg_cotrain - INFO - === Phase 3: Fine-Tuning ===
2026-04-0


Completed in 124.5s
  Test Macro-F1:   0.5626
  Test Error Rate: 22.72%
  Dev Macro-F1:    0.5291


{'task': 'humanitarian',
 'modality': 'image_only',
 'budget': 5,
 'seed_set': 1,
 'test_error_rate': np.float64(22.722513089005236),
 'test_macro_f1': 0.5626397551837623,
 'test_ece': 0.06915059161435874,
 'test_per_class_f1': [0.0,
  0.8227848101265823,
  0.8209982788296041,
  0.7913669064748201,
  0.3780487804878049],
 'dev_error_rate': np.float64(25.751503006012022),
 'dev_macro_f1': 0.5290938510088841,
 'dev_ece': 0.0834236744350804,
 'stopping_strategy': 'baseline',
 'phase1_seed_strategy': 'last',
 'phase1_best_epoch': None,
 'lambda1_mean': 0.8571306998493239,
 'lambda1_std': 0.290931665539737,
 'lambda2_mean': 0.3550877024750314,
 'lambda2_std': 0.1383606979529586}

In [9]:
run_smoke_test("humanitarian", "text_image")


Task: humanitarian | Modality: text_image | Budget: 5 | Seed: 1


emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0
2026-04-01 14:11:15,499 - lg_cotrain - INFO - Starting LG-CoTrain: task=humanitarian, modality=text_image, budget=5, seed_set=1
2026-04-01 14:11:15,548 - lg_cotrain - INFO - Detected 5 classes: ['affected_individuals', 'infrastructure_and_utility_damage', 'not_humanitarian', 'other_relevant_information', 'rescue_volunteering_or_donation_effort']
2026-04-01 14:11:15,551 - lg_cotrain - INFO - Capped unlabeled set to 200 samples
2026-04-01 14:11:15,557 - lg_cotrain - INFO - D_l1: 15, D_l2: 10, D_LG: 200
2026-04-01 14:11:15,559 - lg_cotrain - INFO - === Phase 1: Weight Generation ===
2026-04-01 14:11:23,610 - lg_cotrain - INFO - Phase 1 epoch 1/2: mean_prob1=0.2021, mean_prob2=0.2131
2026-04-01 14:11:29,661 - lg_cotrain - INFO - Phase 1 epoch 2/2: mean_prob1=0.1995, mean_prob2=0.2202
2026-04-01 14:11:29,662 - lg_cotrain - INFO - === Phase 2: Co-Training ===
2026-04-01 14:11:3


Completed in 143.2s
  Test Macro-F1:   0.4083
  Test Error Rate: 26.91%
  Dev Macro-F1:    0.4156


{'task': 'humanitarian',
 'modality': 'text_image',
 'budget': 5,
 'seed_set': 1,
 'test_error_rate': np.float64(26.910994764397913),
 'test_macro_f1': 0.40834471236167447,
 'test_ece': 0.040962699508167666,
 'test_per_class_f1': [0.0,
  0.0,
  0.8257839721254355,
  0.7327935222672065,
  0.48314606741573035],
 'dev_error_rate': np.float64(27.254509018036078),
 'dev_macro_f1': 0.41564254141675094,
 'dev_ece': 0.04657018062227475,
 'stopping_strategy': 'baseline',
 'phase1_seed_strategy': 'last',
 'phase1_best_epoch': None,
 'lambda1_mean': 0.8044015912339378,
 'lambda1_std': 0.24801503317750517,
 'lambda2_mean': 0.3025590901830262,
 'lambda2_std': 0.11286249133315815}

## Summary Table

In [10]:
rows = []
for r in all_results:
    rows.append({
        "Task": r["task"],
        "Modality": r["modality"],
        "Test Macro-F1": f"{r['test_macro_f1']:.4f}",
        "Test Error %": f"{r['test_error_rate']:.2f}",
        "Dev Macro-F1": f"{r['dev_macro_f1']:.4f}",
        "Test ECE": f"{r['test_ece']:.4f}",
        "Lambda1 Mean": f"{r['lambda1_mean']:.4f}",
        "Lambda2 Mean": f"{r['lambda2_mean']:.4f}",
    })

df = pd.DataFrame(rows)
print("Smoke Test Results (minimal epochs — not real performance)")
print("=" * 80)
df

Smoke Test Results (minimal epochs — not real performance)


,Task,Modality,Test Macro-F1,Test Error %,Dev Macro-F1,Test ECE,Lambda1 Mean,Lambda2 Mean
0,informative,text_only,0.4017,32.86,0.4017,0.2030,0.8192,0.4643
1,informative,image_only,0.7819,16.82,0.7694,0.0528,0.9844,0.6090
2,informative,text_image,0.5208,28.88,0.5100,0.2000,0.9563,0.6381
3,humanitarian,text_only,0.1382,47.23,0.1372,0.1061,0.3488,0.2015
4,humanitarian,image_only,0.5626,22.72,0.5291,0.0692,0.8571,0.3551
5,humanitarian,text_image,0.4083,26.91,0.4156,0.0410,0.8044,0.3026


In [11]:
import torch
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    print(f"GPU memory freed. Peak: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")
else:
    print("No GPU available (ran on CPU)")

print(f"\nSmoke test complete. {len(all_results)}/6 experiments succeeded.")

GPU memory freed. Peak: 0.02 GB

Smoke test complete. 6/6 experiments succeeded.
